In [1]:
import torch, sys, os, platform
from ultralytics import __version__ as yv
print("Python :", sys.version.split()[0])
print("OS     :", platform.platform())
print("Torch  :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("Current device   :", torch.cuda.get_device_name(0))
print("Ultralytics:", yv)


Python : 3.12.3
OS     : Linux-6.14.0-27-generic-x86_64-with-glibc2.39
Torch  : 2.3.1+cu121 | CUDA: True
CUDA device count: 1
Current device   : NVIDIA GeForce RTX 4060 Laptop GPU
Ultralytics: 8.3.50


In [2]:
import os, shutil, random, glob
from pathlib import Path

# Point this to your raw export from CVAT if it is NOT split yet
# Expected inside raw_root: "images" and "labels" at the same level, filenames matched
raw_root = Path("datasets/raw")  # change to your folder
images_glob = str(raw_root / "images" / "*.*")
labels_root = raw_root / "labels"

# Target split folders
target_root = Path("datasets/paper_seg2")
for p in ["images/train","images/val","labels/train","labels/val"]:
    (target_root / p).mkdir(parents=True, exist_ok=True)

images = [p for p in glob.glob(images_glob) if p.lower().endswith((".jpg",".jpeg",".png"))]
random.seed(42)
random.shuffle(images)

split = int(0.8 * len(images))
train_imgs = images[:split]
val_imgs   = images[split:]

def move_pair(img_path, split_name):
    img_path = Path(img_path)
    label_path = labels_root / (img_path.stem + ".txt")
    if not label_path.exists():
        print("Missing label for", img_path.name)
        return
    shutil.copy2(img_path, target_root / "images" / split_name / img_path.name)
    shutil.copy2(label_path, target_root / "labels" / split_name / label_path.name)

for p in train_imgs:
    move_pair(p, "train")
for p in val_imgs:
    move_pair(p, "val")

print("Train images:", len(os.listdir(target_root / "images/train")))
print("Val images  :", len(os.listdir(target_root / "images/val")))


Missing label for S__2383901.jpg
Train images: 50
Val images  : 13


In [2]:
from pathlib import Path

root = Path("datasets/paper_seg2")
problems = []

for split in ["train","val"]:
    imdir = root / "images" / split
    lbdir = root / "labels" / split
    for img in imdir.iterdir():
        if img.suffix.lower() not in [".jpg",".jpeg",".png"]:
            continue
        label = lbdir / (img.stem + ".txt")
        if not label.exists():
            problems.append(f"Missing label: {label}")
            continue
        # quick validation: every line should have an odd number of floats after class_id, at least 6 coords
        for ln, line in enumerate(label.read_text().strip().splitlines(), start=1):
            parts = line.strip().split()
            if len(parts) < 8:
                problems.append(f"{label} line {ln}: too few values")
                continue
            try:
                cls = int(parts[0])
                coords = list(map(float, parts[1:]))
            except Exception as e:
                problems.append(f"{label} line {ln}: parse error {e}")
                continue
            if len(coords) % 2 != 0:
                problems.append(f"{label} line {ln}: odd number of coords")
            if cls != 0:
                problems.append(f"{label} line {ln}: class must be 0 for single-class dataset")

print("Problems found:" if problems else "All good.")
for p in problems[:20]:
    print("-", p)
if len(problems) > 20:
    print("... more omitted")


All good.


In [7]:
from pathlib import Path
yaml_text = """\
path: datasets/paper_seg2
train: images/train
val: images/val

names:
  0: paper
"""
Path("paper_seg.yaml").write_text(yaml_text)
print(Path("paper_seg.yaml").resolve())
print("---")
print(yaml_text)


/media/tameszaza/MyPassport/ml/OPH_project/2025/paper_seg.yaml
---
path: datasets/paper_seg2
train: images/train
val: images/val

names:
  0: paper



In [ ]:
model_name = "yolo11n-seg.pt"   # try "yolov8m-seg.pt" for better accuracy
epochs     = 500
imgsz      = 640
batch      = 16                  # adjust to your GPU memory
project    = "runs_paper_seg"
name       = "y11n_baseline"


In [3]:
# You can tweak these later
augment_args = dict(
    degrees=5.0,        # small rotations
    translate=0.05,     # small shifts
    scale=0.10,         # zoom in/out
    shear=4.0,
    perspective=0.000,  # set >0 if you want projective transforms
    flipud=0.0,
    fliplr=0.5,         # horizontal flips
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4
)
augment_args


{'degrees': 5.0,
 'translate': 0.05,
 'scale': 0.1,
 'shear': 4.0,
 'perspective': 0.0,
 'flipud': 0.0,
 'fliplr': 0.5,
 'hsv_h': 0.015,
 'hsv_s': 0.7,
 'hsv_v': 0.4}

In [4]:
from ultralytics import YOLO

model = YOLO(model_name)
# If you want to start from scratch instead of a pretrained backbone, use:
# model = YOLO("yolov8n-seg.yaml")  # then model.train(...)

results = model.train(
    data="paper_seg.yaml",
    epochs=epochs,
    imgsz=imgsz,
    batch=batch,
    project=project,
    name=name,
    device=0 if torch.cuda.is_available() else "cpu",
    # augmentation
    degrees=augment_args["degrees"],
    translate=augment_args["translate"],
    scale=augment_args["scale"],
    shear=augment_args["shear"],
    perspective=augment_args["perspective"],
    flipud=augment_args["flipud"],
    fliplr=augment_args["fliplr"],
    hsv_h=augment_args["hsv_h"],
    hsv_s=augment_args["hsv_s"],
    hsv_v=augment_args["hsv_v"],
    # optimization niceties
    patience=50,        # early stop if val metric stalls
    cos_lr=True,
    workers=2           # set higher on Linux if disk is fast
)
print("Best weights in:", results.save_dir)


New https://pypi.org/project/ultralytics/8.3.176 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.50 🚀 Python-3.12.3 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
engine/trainer: task=segment, mode=train, model=yolov8m-seg.pt, data=paper_seg.yaml, epochs=500, time=None, patience=50, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=2, project=runs_paper_seg, name=y8m_baseline3, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embe

train: Scanning /media/tameszaza/MyPassport/ml/OPH_project/2025/datasets/paper_seg2/labels/train.cache... 50 images, 0 backgrounds, 0 corrupt: 100%|██████████| 50/50 [00:00<?, ?it/s]
val: Scanning /media/tameszaza/MyPassport/ml/OPH_project/2025/datasets/paper_seg2/labels/val.cache... 13 images, 0 backgrounds, 0 corrupt: 100%|██████████| 13/13 [00:00<?, ?it/s]


Plotting labels to runs_paper_seg/y8m_baseline3/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 86 weight(decay=0.0), 97 weight(decay=0.0005), 96 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to runs_paper_seg/y8m_baseline3
Starting training for 500 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


      1/500      7.81G     0.9612      2.061      2.516      1.541          4        640: 100%|██████████| 4/4 [00:03<00:00,  1.30it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  4.40it/s]

                   all         13         13      0.825      0.366      0.493      0.456      0.825      0.366      0.493      0.492


...

    250/500      7.74G     0.4182     0.1215     0.3214      1.025          4        640: 100%|██████████| 4/4 [00:02<00:00,  1.61it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  4.81it/s]

                   all         13         13      0.994          1      0.995      0.978      0.994          1      0.995      0.995
EarlyStopping: Training stopped early as no improvement observed in last 50 epochs. Best results observed at epoch 200, best model saved as best.pt.
To update EarlyStopping(patience=50) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



250 epochs completed in 0.272 hours.
Optimizer stripped from runs_paper_seg/y8m_baseline3/weights/last.pt, 54.9MB
Optimizer stripped from runs_paper_seg/y8m_baseline3/weights/best.pt, 54.9MB

Validating runs_paper_seg/y8m_baseline3/weights/best.pt...
Ultralytics 8.3.50 🚀 Python-3.12.3 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
YOLOv8m-seg summary (fused): 245 layers, 27,222,963 parameters, 0 gradients, 110.0 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  5.06it/s]


                   all         13         13      0.993          1      0.995      0.995      0.993          1      0.995      0.995
Speed: 0.2ms preprocess, 11.2ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to runs_paper_seg/y8m_baseline3
Best weights in: runs_paper_seg/y8m_baseline3


In [ ]:
# This will use the best.pt from training
best = f"{results.save_dir}/weights/best.pt"
val_res = YOLO(best).val(data="paper_seg.yaml", imgsz=imgsz, batch=batch)
val_res


Ultralytics 8.3.176 🚀 Python-3.12.3 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7806MiB)
YOLOv8n-seg summary (fused): 85 layers, 3,258,259 parameters, 0 gradients, 12.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2825.3±291.7 MB/s, size: 303.6 KB)


val: Scanning /media/tameszaza/MyPassport/ml/OPH_project/2025/datasets/paper_seg/labels/val.cache... 5 images, 0 backgrounds, 0 corrupt: 100%|██████████| 5/5 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95):   0%|          | 0/1 [00:00<?, ?it/s]

WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  5.94it/s]


                   all          5          5      0.991          1      0.995      0.975      0.991          1      0.995      0.995
Speed: 0.2ms preprocess, 26.4ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /media/tameszaza/MyPassport/ml/OPH_project/runs/segment/val


ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x797c341c8380>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041, 

In [ ]:
import cv2, numpy as np, glob, os
from pathlib import Path
from ultralytics import YOLO

best = f"{results.save_dir}/weights/best.pt"
model = YOLO(best)

src_dir = Path("datasets/paper_seg/images/val")
out_dir = Path("cutouts")
out_dir.mkdir(exist_ok=True)

for img_path in sorted(src_dir.glob("*.*")):
    if img_path.suffix.lower() not in [".jpg",".jpeg",".png"]:
        continue
    preds = model.predict(source=str(img_path), imgsz=imgsz, conf=0.3, iou=0.5, verbose=False)
    if not preds:
        continue
    pred = preds[0]
    if pred.masks is None or pred.masks.data.shape[0] == 0:
        continue

    # pick top confidence box/mask for class 0
    confs = pred.boxes.conf.cpu().numpy()
    classes = pred.boxes.cls.cpu().numpy().astype(int)
    idx = None
    best_conf = -1
    for i, (c, cf) in enumerate(zip(classes, confs)):
        if c == 0 and cf > best_conf:
            best_conf = cf
            idx = i
    if idx is None:
        continue

    mask = pred.masks.data[idx].cpu().numpy().astype(np.uint8) * 255  # [H,W]
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    h, w = img.shape[:2]
    if mask.shape != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

    result = cv2.bitwise_and(img, img, mask=mask)
    b,g,r = cv2.split(result)
    rgba = cv2.merge((b,g,r,mask))
    out_file = out_dir / (img_path.stem + "_paper.png")
    cv2.imwrite(str(out_file), rgba)

print("PNG cutouts saved to:", out_dir)


NameError: name 'results' is not defined

In [ ]:
import cv2
import numpy as np
from math import sqrt

def clean_mask_remove_islands(mask_uint8, keep_frac=0.02, min_area_px=500, close_ks=5):
    """
    Input:  mask_uint8 in {0,255}
    Output: cleaned mask in {0,255} with small connected components removed

    keep_frac: keep any component whose area >= keep_frac * area_of_largest_component
    min_area_px: also keep any component larger than this absolute area
    close_ks: kernel size for a closing op to fill small holes along the edge
    """
    if mask_uint8.dtype != np.uint8:
        mask_uint8 = mask_uint8.astype(np.uint8)
    mask_bin = (mask_uint8 > 127).astype(np.uint8) * 255

    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask_bin, connectivity=8)
    if num <= 1:
        return mask_bin

    # find the largest non-background component
    areas = stats[1:, cv2.CC_STAT_AREA]  # drop background row 0
    largest_idx = 1 + np.argmax(areas)
    largest_area = stats[largest_idx, cv2.CC_STAT_AREA]

    keep = np.zeros_like(labels, dtype=np.uint8)
    for cid in range(1, num):
        area = stats[cid, cv2.CC_STAT_AREA]
        if area >= max(int(keep_frac * largest_area), int(min_area_px)):
            keep[labels == cid] = 255

    # optional morphological closing to fill pinholes and smooth edges
    if close_ks and close_ks > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_ks, close_ks))
        keep = cv2.morphologyEx(keep, cv2.MORPH_CLOSE, k, iterations=1)

    return keep


def order_points_clockwise(pts):
    # pts: (4,2) float32
    c = np.mean(pts, axis=0)
    angles = np.arctan2(pts[:,1] - c[1], pts[:,0] - c[0])
    idx = np.argsort(angles)
    ordered = pts[idx]
    # ensure first is top-left by y then x
    tl_idx = np.argmin(ordered[:,1] + 0.3*ordered[:,0])
    ordered = np.roll(ordered, -tl_idx, axis=0)
    return ordered.astype(np.float32)


def quad_from_mask(mask_uint8, approx_eps_frac=0.02):
    """
    Try to get 4 corners from the biggest contour.
    If approx does not give 4 points, fall back to minAreaRect.
    Returns 4x2 float32 points.
    """
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    cnt = max(contours, key=cv2.contourArea)
    peri = cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, epsilon=approx_eps_frac * peri, closed=True)

    if len(approx) == 4:
        quad = approx.reshape(-1, 2).astype(np.float32)
    else:
        # min area rectangle fallback
        rect = cv2.minAreaRect(cnt)
        box = cv2.boxPoints(rect)  # 4x2
        quad = box.astype(np.float32)

    return order_points_clockwise(quad)


def compute_output_size(quad, force_a4=False):
    """
    Decide output W,H from the quad.
    If force_a4 is True, enforce ISO ratio (1:sqrt(2)) in either orientation while preserving the longer edge length.
    """
    def dist(a,b): return np.linalg.norm(a-b)

    tl, tr, br, bl = quad
    w_top  = dist(tl, tr)
    w_bot  = dist(bl, br)
    h_left = dist(tl, bl)
    h_right= dist(tr, br)

    W = int(round(max(w_top, w_bot)))
    H = int(round(max(h_left, h_right)))

    W = max(W, 10)
    H = max(H, 10)

    if not force_a4:
        return W, H

    # ISO A ratio: longer/shorter = sqrt(2)
    ratio = sqrt(2)
    if W >= H:
        # landscape
        target_W = W
        target_H = int(round(target_W / ratio))
    else:
        # portrait
        target_H = H
        target_W = int(round(target_H / ratio))

    # keep sizes reasonable
    target_W = max(64, target_W)
    target_H = max(64, target_H)
    return target_W, target_H


def warp_perspective(image_bgr, mask_uint8, force_a4=False, eps_frac=0.02):
    """
    Clean mask, detect quad, warp image to a rectangle.
    Returns rectified_bgr, cleaned_mask, quad_points
    """
    clean = clean_mask_remove_islands(mask_uint8)
    quad = quad_from_mask(clean, approx_eps_frac=eps_frac)
    if quad is None:
        return None, clean, None

    W, H = compute_output_size(quad, force_a4=force_a4)
    dst = np.array([[0,0],[W-1,0],[W-1,H-1],[0,H-1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(quad, dst)
    rectified = cv2.warpPerspective(image_bgr, M, (W, H), flags=cv2.INTER_LINEAR)
    return rectified, clean, quad


In [2]:
import cv2
import numpy as np
from math import sqrt
from skimage.transform import estimate_transform, warp

# ---------- helpers ----------
def clean_mask_remove_islands(mask_uint8, keep_frac=0.02, min_area_px=800, close_ks=5, dilate=0):
    if mask_uint8.dtype != np.uint8:
        mask_uint8 = mask_uint8.astype(np.uint8)
    binm = (mask_uint8 > 127).astype(np.uint8) * 255
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binm, connectivity=8)
    if num <= 1:
        out = binm
    else:
        areas = stats[1:, cv2.CC_STAT_AREA]
        largest = areas.max() if len(areas) else 0
        keep = np.zeros_like(labels, dtype=np.uint8)
        for cid in range(1, num):
            a = stats[cid, cv2.CC_STAT_AREA]
            if a >= max(int(keep_frac*largest), int(min_area_px)):
                keep[labels == cid] = 255
        out = keep
    if close_ks and close_ks > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_ks, close_ks))
        out = cv2.morphologyEx(out, cv2.MORPH_CLOSE, k, iterations=1)
    if dilate > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate, dilate))
        out = cv2.dilate(out, k, iterations=1)
    return out

def largest_contour(mask_uint8):
    cnts, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    return max(cnts, key=cv2.contourArea).reshape(-1, 2).astype(np.float32)

def resample_closed_contour(contour, n_pts):
    """Uniform arc-length resample of a CLOSED contour."""
    pts = contour
    seg = np.linalg.norm(np.diff(np.vstack([pts, pts[:1]]), axis=0), axis=1)
    arclen = np.concatenate([[0], np.cumsum(seg)])
    total = arclen[-1]
    if total < 1e-6:
        return np.repeat(pts[:1], n_pts, axis=0)
    targets = np.linspace(0, total, n_pts+1)[:-1]  # drop duplicate last point
    res = []
    j = 0
    for t in targets:
        while j+1 < len(arclen) and arclen[j+1] < t:
            j += 1
        j2 = (j+1) % len(pts)
        seglen = arclen[j+1] - arclen[j]
        a = 0.0 if seglen < 1e-8 else (t - arclen[j]) / seglen
        res.append(pts[j]*(1-a) + pts[j2]*a)
    return np.array(res, dtype=np.float32)

def compute_rect_size_from_contour(contour, force_a4=False):
    # rough size from bounding box edges
    x, y, w, h = cv2.boundingRect(contour.astype(np.int32))
    W = max(int(round(w)), 64)
    H = max(int(round(h)), 64)
    if not force_a4:
        return W, H
    r = sqrt(2)
    if W >= H:
        H = int(round(W / r))
    else:
        W = int(round(H / r))
    return max(64, W), max(64, H)

def rectangle_perimeter_points(W, H, n_pts):
    """Return n_pts samples along the perimeter of a W x H rectangle starting at TL and going clockwise."""
    per = 2*(W+H)
    targets = np.linspace(0, per, n_pts, endpoint=False)
    pts = []
    for t in targets:
        s = t % per
        if s < W:                           # top edge: (0,0) -> (W,0)
            x = s; y = 0
        elif s < W + H:                     # right edge: (W,0) -> (W,H)
            x = W; y = s - W
        elif s < W + H + W:                 # bottom edge: (W,H) -> (0,H)
            x = W - (s - (W+H)); y = H
        else:                                # left edge: (0,H) -> (0,0)
            x = 0; y = H - (s - (W+H+W))
        pts.append([x, y])
    return np.array(pts, dtype=np.float32)
import cv2
import numpy as np
from math import sqrt

# keep your clean_mask_remove_islands, largest_contour, resample_closed_contour,
import cv2
import numpy as np
from math import sqrt

# assumes you already have:
# clean_mask_remove_islands, largest_contour, resample_closed_contour,
# compute_rect_size_from_contour, rectangle_perimeter_points

def _make_tps(regularization: float):
    # try top-level API (older contrib builds)
    if hasattr(cv2, "createThinPlateSplineShapeTransformer"):
        return cv2.createThinPlateSplineShapeTransformer(regularization=regularization)
    # try ximgproc (newer contrib builds)
    try:
        import cv2.ximgproc as xip
        return xip.createThinPlateSplineShapeTransformer(regularization)
    except Exception as e:
        raise RuntimeError(
            "Your OpenCV build lacks TPS. Install contrib build:\n"
            "pip uninstall -y opencv-python && pip install opencv-contrib-python==4.9.0.80"
        ) from e

def dewarp_document_tps(image_bgr, mask_uint8,
                        n_border_samples=600,
                        force_a4=False,
                        tps_smooth=1e-3,
                        output_scale=1.2,
                        background_color=255):
    """
    Non-rigid dewarp using OpenCV Thin Plate Spline by matching the full page border to a rectangle border.
    Returns rectified_bgr, rect_alpha, cleaned_mask, debug
    """
    H0, W0 = image_bgr.shape[:2]

    # 1) clean mask
    clean = clean_mask_remove_islands(mask_uint8, keep_frac=0.02, min_area_px=1500, close_ks=5, dilate=1)

    # 2) largest contour
    cnt = largest_contour(clean)
    if cnt is None or len(cnt) < 30:
        return None, None, clean, {"reason": "no usable contour"}

    # small smoothing
    cnt_s = cv2.GaussianBlur(cnt.astype(np.float32), (0, 0), 1.0)

    # 3) resample full closed border
    src_border = resample_closed_contour(cnt_s, n_border_samples)  # xy

    # 4) destination rectangle perimeter
    Wrect, Hrect = compute_rect_size_from_contour(cnt_s, force_a4=force_a4)
    Wrect = max(64, int(round(Wrect * output_scale)))
    Hrect = max(64, int(round(Hrect * output_scale)))
    dst_border = rectangle_perimeter_points(Wrect - 1, Hrect - 1, n_border_samples)  # xy

    # 5) TPS transformer
    tps = _make_tps(tps_smooth)

    fromShape = src_border.reshape(-1, 1, 2).astype(np.float32)  # page border (xy)
    toShape   = dst_border.reshape(-1, 1, 2).astype(np.float32)  # rect border (xy)
    matches   = [cv2.DMatch(i, i, 0) for i in range(n_border_samples)]

    tps.estimateTransformation(fromShape, toShape, matches)

    # 6) warp image (white background)
    rect_bgr = tps.warpImage(
        image_bgr,
        (Wrect, Hrect),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(int(background_color), int(background_color), int(background_color))
    )

    # 7) warp mask to alpha
    if clean.ndim == 3:
        clean = cv2.cvtColor(clean, cv2.COLOR_BGR2GRAY)
    rect_mask = tps.warpImage(
        clean,
        (Wrect, Hrect),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )
    rect_alpha = rect_mask if rect_mask.ndim == 2 else rect_mask[:, :, 0]

    debug = {
        "src_border": src_border,
        "dst_border": dst_border,
        "out_size": (Wrect, Hrect)
    }
    return rect_bgr, rect_alpha, clean, debug




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/media/tameszaza/MyPassport/ml/OPH_project/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/media/tameszaza/MyPassport/ml/OPH_project/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/media/tameszaza/MyPassport/ml/OPH_project/.venv/lib/python3.12/site-packages/ipykernel/

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [3]:
import os
import glob
from pathlib import Path
from math import sqrt

import cv2
import numpy as np
from ultralytics import YOLO


# =========================
# Mask utilities
# =========================
def clean_mask_remove_islands(mask_uint8, keep_frac=0.02, min_area_px=1500, close_ks=5, dilate=1):
    """
    Keep only large connected components from a binary mask and optionally close tiny gaps.
    mask_uint8: 0..255
    """
    if mask_uint8.dtype != np.uint8:
        mask_uint8 = mask_uint8.astype(np.uint8)
    binm = (mask_uint8 > 127).astype(np.uint8) * 255

    num, labels, stats, _ = cv2.connectedComponentsWithStats(binm, connectivity=8)
    if num <= 1:
        out = binm
    else:
        areas = stats[1:, cv2.CC_STAT_AREA]
        largest = areas.max() if len(areas) else 0
        keep = np.zeros_like(labels, dtype=np.uint8)
        for cid in range(1, num):
            a = stats[cid, cv2.CC_STAT_AREA]
            if a >= max(int(keep_frac * largest), int(min_area_px)):
                keep[labels == cid] = 255
        out = keep

    if close_ks and close_ks > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_ks, close_ks))
        out = cv2.morphologyEx(out, cv2.MORPH_CLOSE, k, iterations=1)
    if dilate and dilate > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate, dilate))
        out = cv2.dilate(out, k, iterations=1)
    return out


def largest_contour(mask_uint8):
    cnts, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    return max(cnts, key=cv2.contourArea).reshape(-1, 2).astype(np.float32)


def resample_closed_contour(contour, n_pts):
    """Uniform arc-length resample of a CLOSED contour to n_pts points."""
    pts = contour.astype(np.float32)
    seg = np.linalg.norm(np.diff(np.vstack([pts, pts[:1]]), axis=0), axis=1)
    arclen = np.concatenate([[0], np.cumsum(seg)])
    total = arclen[-1]
    if total < 1e-6:
        return np.repeat(pts[:1], n_pts, axis=0)
    targets = np.linspace(0, total, n_pts, endpoint=False)
    res = []
    j = 0
    for t in targets:
        while j + 1 < len(arclen) and arclen[j + 1] < t:
            j += 1
        j2 = (j + 1) % len(pts)
        seglen = arclen[j + 1] - arclen[j]
        a = 0.0 if seglen < 1e-8 else (t - arclen[j]) / seglen
        res.append(pts[j] * (1 - a) + pts[j2] * a)
    return np.array(res, dtype=np.float32)


def compute_rect_size_from_contour(contour, force_a4=False):
    x, y, w, h = cv2.boundingRect(contour.astype(np.int32))
    W = max(int(round(w)), 64)
    H = max(int(round(h)), 64)
    if not force_a4:
        return W, H
    r = sqrt(2)
    if W >= H:
        H = int(round(W / r))
    else:
        W = int(round(H / r))
    return max(64, W), max(64, H)


def rectangle_perimeter_points(W, H, n_pts):
    """n_pts samples along the perimeter of a W x H rectangle starting TL clockwise."""
    per = 2 * (W + H)
    targets = np.linspace(0, per, n_pts, endpoint=False)
    pts = []
    for s in targets:
        if s < W:
            x = s; y = 0
        elif s < W + H:
            x = W; y = s - W
        elif s < W + H + W:
            x = W - (s - (W + H)); y = H
        else:
            x = 0; y = H - (s - (W + H + W))
        pts.append([x, y])
    return np.array(pts, dtype=np.float32)


# =========================
# TPS warp
# =========================
def _make_tps(regularization: float):
    # OpenCV 4.10.0 has top-level TPS
    if hasattr(cv2, "createThinPlateSplineShapeTransformer"):
        return cv2.createThinPlateSplineShapeTransformer(regularization=regularization)
    # Some builds ship it under ximgproc
    try:
        import cv2.ximgproc as xip
        return xip.createThinPlateSplineShapeTransformer(regularization)
    except Exception as e:
        raise RuntimeError("Thin Plate Spline not available in your OpenCV build") from e


def dewarp_document_tps(
    image_bgr,
    mask_uint8,
    n_border_samples=800,
    force_a4=False,
    tps_smooth=5e-3,
    output_scale=1.25,
    background_color=255
):
    """
    Non-rigid dewarp using OpenCV TPS by matching full page border -> rectangle border.
    Returns rectified_bgr, rect_alpha, cleaned_mask, debug
    """
    H0, W0 = image_bgr.shape[:2]

    # 1) clean and keep largest component
    clean = clean_mask_remove_islands(mask_uint8, keep_frac=0.02, min_area_px=1500, close_ks=5, dilate=1)

    # 2) largest contour
    cnt = largest_contour(clean)
    if cnt is None or len(cnt) < 30:
        return None, None, clean, {"reason": "no usable contour"}

    # 3) mild smoothing and uniform resample of full closed border
    cnt_s = cv2.GaussianBlur(cnt, (0, 0), 1.0)
    src_border = resample_closed_contour(cnt_s, n_border_samples)  # [N,2] xy

    # 4) choose output canvas and build rectangle border with same samples
    Wrect, Hrect = compute_rect_size_from_contour(cnt_s, force_a4=force_a4)
    Wrect = max(64, int(round(Wrect * output_scale)))
    Hrect = max(64, int(round(Hrect * output_scale)))
    dst_border = rectangle_perimeter_points(Wrect - 1, Hrect - 1, n_border_samples)  # [N,2] xy

    # 5) TPS estimate from page->rectangle
    tps = _make_tps(tps_smooth)
    fromShape = src_border.reshape(-1, 1, 2).astype(np.float32)  # xy
    toShape = dst_border.reshape(-1, 1, 2).astype(np.float32)    # xy
    matches = [cv2.DMatch(i, i, 0) for i in range(n_border_samples)]
    tps.estimateTransformation(fromShape, toShape, matches)

    # 6) warp image and mask
    rect_bgr = tps.warpImage(
        image_bgr,
        (Wrect, Hrect),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(int(background_color),) * 3
    )
    rect_mask = tps.warpImage(
        clean,
        (Wrect, Hrect),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )
    rect_alpha = rect_mask if rect_mask.ndim == 2 else rect_mask[:, :, 0]

    debug = {"src_border": src_border, "dst_border": dst_border, "out_size": (Wrect, Hrect)}
    return rect_bgr, rect_alpha, clean, debug


# =========================
# YOLO inference + dewarp
# =========================
def pick_latest_best(weights_glob="runs_paper_seg/*/weights/best.pt"):
    found = sorted(glob.glob(weights_glob))
    if not found:
        raise FileNotFoundError(f"No weights found under {weights_glob}")
    return found[-1]


def run_folder(
    model_path,
    src_dir="datasets/paper_seg/images/val",
    out_dir="docmatcher",
    imgsz=960,
    conf=0.25,
    iou=0.5,
    n_border_samples=800,
    tps_smooth=5e-3,
    force_a4=False,
    output_scale=1.25
):
    model = YOLO(model_path)
    src_dir = Path(src_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for img_path in sorted(src_dir.glob("*.*")):
        if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            print("skip unreadable:", img_path.name)
            continue
        H, W = img.shape[:2]

        preds = model.predict(source=str(img_path), imgsz=imgsz, conf=conf, iou=iou, verbose=False)
        if not preds or preds[0].masks is None or preds[0].masks.data.shape[0] == 0:
            print("no mask:", img_path.name)
            continue

        # pick highest-confidence mask of class 0
        boxes = preds[0].boxes
        cls = boxes.cls.cpu().numpy().astype(int)
        confs = boxes.conf.cpu().numpy()
        idx = None
        bestc = -1.0
        for i, (c, cf) in enumerate(zip(cls, confs)):
            if c == 0 and cf > bestc:
                idx = i; bestc = cf
        if idx is None:
            print("no class 0:", img_path.name)
            continue

        raw_mask = preds[0].masks.data[idx].cpu().numpy().astype(np.uint8) * 255
        if raw_mask.shape != (H, W):
            raw_mask = cv2.resize(raw_mask, (W, H), interpolation=cv2.INTER_NEAREST)

        rect_bgr, rect_alpha, clean_mask, dbg = dewarp_document_tps(
            img, raw_mask,
            n_border_samples=n_border_samples,
            force_a4=force_a4,
            tps_smooth=tps_smooth,
            output_scale=output_scale,
            background_color=255
        )
        if rect_bgr is None:
            print("dewarp failed:", img_path.name, dbg.get("reason", ""))
            continue

        base = img_path.stem
        # save PNG with alpha and a JPG preview
        b, g, r = cv2.split(rect_bgr)
        rgba = cv2.merge((b, g, r, rect_alpha))
        cv2.imwrite(str(out_dir / f"{base}_paper.png"), rgba)
        cv2.imwrite(str(out_dir / f"{base}_rectified.jpg"), rect_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 92])

        # optional debug
        cv2.imwrite(str(out_dir / f"{base}_mask_clean.png"), clean_mask)
        vis = img.copy()
        for p in dbg["src_border"].astype(int)[::4]:
            cv2.circle(vis, tuple(p), 2, (0, 255, 0), -1)
        cv2.imwrite(str(out_dir / f"{base}_border_debug.jpg"), vis)

        print(f"ok: {img_path.name} -> {base}_paper.png")


if __name__ == "__main__":
    # auto-pick latest trained weights
    best = pick_latest_best("runs_paper_seg/*/weights/best.pt")
    print("Using weights:", best)
    run_folder(
        model_path=best,
        src_dir="datasets/paper_seg/images/val",
        out_dir="docmatcher",
        imgsz=960,
        conf=0.25,
        iou=0.5,
        n_border_samples=800,
        tps_smooth=5e-3,
        force_a4=False,      # set True if you always want ISO-A aspect ratio
        output_scale=1.25    # bump to 1.30 if edges clip
    )



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/media/tameszaza/MyPassport/ml/OPH_project/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/media/tameszaza/MyPassport/ml/OPH_project/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/media/tameszaza/MyPassport/ml/OPH_project/.venv/lib/python3.12/site-packages/ipykernel/

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [1]:
from ultralytics import YOLO



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/media/tameszaza/MyPassport/ml/OPH_project/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/media/tameszaza/MyPassport/ml/OPH_project/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/media/tameszaza/MyPassport/ml/OPH_project/.venv/lib/python3.12/site-packages/ipykernel/

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import